In [ ]:
import torch
import numpy as np
import wave
import os
import time
import threading
import subprocess
import platform
from TTS.api import TTS
from queue import Queue, Empty  # Import Empty specifically

# Create output directory
output_dir = "tts_chunks"
os.makedirs(output_dir, exist_ok=True)

# Initialize TTS
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(TTS().list_models())

# Load model
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

sample_rate = tts.synthesizer.tts_config.audio["sample_rate"]
print(f"Sample rate: {sample_rate}")


In [ ]:

# Text to synthesize
text = """
Es war einmal eine kleine Wolke namens Cira, die nicht wusste, wie man regnet. Sie schwebte allein am Himmel und beobachtete, wie die anderen Wolken die Erde bewässerten.
Eines Tages sah sie eine weise alte Wolke und sagte: „Cira, du musst dein Herz mit Güte füllen. Wenn du voller Liebe bist, wird der Regen ganz von selbst kommen.“
Cira verbrachte Tage damit, Blumen beim Blühen, Kinder beim Lachen und Flüsse beim Fließen zu beobachten. Ihr Herz wurde immer wärmer.
Dann, an einem Nachmittag, spürte sie etwas in sich. Ein kleiner Tropfen fiel… dann noch einer… bis sie zum allerersten Mal regnete und einem ausgetrockneten Dorf unten Freude brachte.
Von da an regnete Cira mit Freude und Sinn – nie zu wenig, nie zu viel.
"""

speaker_file = "default_audio/audio1.wav"
language = "de"

print(f"Generating speech for: '{text}'")
print(f"Using speaker file: {speaker_file}")
print(f"Language: {language}")

# Function to save audio to WAV file
def save_audio(audio_chunk, filename, sr=22050):
    # Convert to numpy array if needed
    if not isinstance(audio_chunk, np.ndarray):
        audio_chunk = np.array(audio_chunk)
    
    # Normalize and convert to int16
    audio_chunk = audio_chunk / np.max(np.abs(audio_chunk)) * 32767
    audio_chunk = audio_chunk.astype(np.int16)
    
    # Save as WAV file
    with wave.open(filename, 'wb') as wf:
        wf.setnchannels(1)  # Mono
        wf.setsampwidth(2)  # 16-bit
        wf.setframerate(sr)
        wf.writeframes(audio_chunk.tobytes())
    
    return filename

# Function to play audio with system commands
def play_audio_file(filename):
    system = platform.system()
    
    try:
        if system == 'Darwin':  # macOS
            subprocess.call(['afplay', filename])
        elif system == 'Linux':
            subprocess.call(['aplay', filename])
        elif system == 'Windows':
            subprocess.call(['powershell', '-c', f'(New-Object Media.SoundPlayer "{filename}").PlaySync();'])
        print(f"Played {filename}")
    except Exception as e:
        print(f"Error playing audio: {e}")

# Create a queue to communicate between threads
audio_queue = Queue()
processing_done = threading.Event()

# Function for player thread
def player_thread_function():
    full_audio = []
    chunk_count = 0
    
    while not (processing_done.is_set() and audio_queue.empty()):
        try:
            # Get the next audio chunk from the queue with a timeout
            # The timeout allows checking the processing_done flag periodically
            item = audio_queue.get(timeout=0.5)
            
            # Save the audio chunk
            chunk_count += 1
            chunk_filename = os.path.join(output_dir, f"chunk_{chunk_count}.wav")
            save_audio(item, chunk_filename, sample_rate)
            
            # Play the audio file
            print(f"Playing chunk {chunk_count}...")
            play_audio_file(chunk_filename)
            
            # Append to full audio for combined file
            full_audio.extend(item)
            
            # Mark this task as done
            audio_queue.task_done()
            
        except Empty:  # Correctly catching the Empty exception now
            # Queue is empty but processing might not be done yet
            continue
    
    # Save combined audio when done
    if full_audio:
        combined_filename = os.path.join(output_dir, "combined.wav")
        save_audio(full_audio, combined_filename, sample_rate)
        print(f"Finished generating {chunk_count} audio chunks")
        print(f"Audio files saved to {os.path.abspath(output_dir)}")

# Start the player thread
player_thread = threading.Thread(target=player_thread_function)
player_thread.daemon = True  # Thread will exit when main program exits
player_thread.start()

try:
    # Split the text into natural sentences
    sentences = [text]
    print(f"Processing {len(sentences)} sentences: {sentences}")
    
    for sentence in sentences:
        # Generate audio for each sentence
        t = tts.synthesizer.tts_stream(
            text=sentence, 
            speaker_wav=speaker_file, 
            language_name=language
        )
        for chunk in t:
            # Add the chunk to the queue for processing
            audio_queue.put(chunk)
            
        # Add a small pause between sentences (by adding silence)
        silence_duration = 0.3  # seconds
        silence_samples = int(silence_duration * sample_rate)
        silence = np.zeros(silence_samples, dtype=np.float32)
        audio_queue.put(silence)
    
    # Signal that we're done generating
    processing_done.set()
    
    # Wait for all items to be processed
    # Add a timeout to audio_queue.join() in case it gets stuck
    try:
        audio_queue.join()
    except Exception as e:
        print(f"Queue join error: {e}")
    
    # Add this to ensure we exit properly
    print("Processing complete, exiting...")
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()
finally:
    # Ensure we set the flag in case of exceptions
    processing_done.set()

In [ ]:
import requests
import logging
import time

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [%(levelname)s] - %(message)s'
)

def test_tts_api(ip="135.181.71.42", port=8004):
    base_url = f"http://{ip}:{port}"
    endpoint = "/synthesize"
    full_url = f"{base_url}{endpoint}"
    
    # Sample query parameters
    params = {
        "text": "Hello, this is a test.",
        "language": "en"
    }

    logging.info(f"Testing FastAPI TTS endpoint at {full_url}")
    logging.info(f"Query params: {params}")

    try:
        logging.info("Sending GET request to the API...")
        response = requests.get(full_url, params=params, stream=True, timeout=15)

        logging.info(f"Received HTTP status code: {response.status_code}")
        if response.status_code == 200:
            logging.info("API is accessible ✅")
            
            # Read first few bytes of the stream
            logging.info("Reading stream to verify audio content...")
            audio_chunk = next(response.iter_content(chunk_size=1024), None)
            if audio_chunk:
                logging.info(f"Received {len(audio_chunk)} bytes of audio data.")
            else:
                logging.warning("No audio content received from stream ❗")
        else:
            logging.error(f"Non-200 response: {response.status_code}")
            logging.error(f"Response content: {response.text}")
    except requests.exceptions.RequestException as e:
        logging.error(f"Request failed: {e}")

if __name__ == "__main__":
    start_time = time.time()
    test_tts_api()
    logging.info(f"Test completed in {time.time() - start_time:.2f} seconds")
